# NGIML Inference (Colab)
Load a trained checkpoint and run a sanity-check inference on one sample from the manifest.

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/DeogenesMaranan/ngiml"
REPO_DIR = Path("/content/ngiml")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

sys.path.insert(0, str(REPO_DIR))
print("Repo ready at", REPO_DIR)

In [ ]:
from google.colab import drive
from pathlib import Path

DRIVE_MOUNT = "/content/drive"
RUNS_DIR = Path(f"{DRIVE_MOUNT}/MyDrive/ngiml_runs")
DATA_DIR = Path("/content/data")

drive.mount(DRIVE_MOUNT)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print("Runs dir:", RUNS_DIR)
print("Data dir:", DATA_DIR)

In [ ]:
import os
import torch
from pathlib import Path

from tools.colab_train_helpers import find_or_resolve_manifest
from tools.local_infer_helpers import load_model_from_checkpoint
from src.data.dataloaders import load_manifest

# Prefer the IoU-best checkpoint for overlap quality; fallback to best monitored checkpoint, then latest epoch.
best_iou_candidates = sorted(RUNS_DIR.rglob("checkpoints/best_iou_checkpoint.pt"), key=lambda p: p.stat().st_mtime)
if best_iou_candidates:
    CKPT_PATH = best_iou_candidates[-1]
else:
    best_ckpt_candidates = sorted(RUNS_DIR.rglob("checkpoints/best_checkpoint.pt"), key=lambda p: p.stat().st_mtime)
    if best_ckpt_candidates:
        CKPT_PATH = best_ckpt_candidates[-1]
    else:
        ckpt_candidates = sorted(RUNS_DIR.rglob("checkpoints/checkpoint_epoch_*.pt"), key=lambda p: p.stat().st_mtime)
        if not ckpt_candidates:
            raise FileNotFoundError(f"No checkpoint found under {RUNS_DIR}/**/checkpoints/")
        CKPT_PATH = ckpt_candidates[-1]
print("Using checkpoint:", CKPT_PATH)

# Resolve manifest paths for Colab runtime (rewrites host-local paths to current runtime paths).
try:
    MANIFEST_PATH = find_or_resolve_manifest(DATA_DIR)
except FileNotFoundError:
    print(f"No manifest found under {DATA_DIR}. Attempting dataset download...")
    from huggingface_hub import login, snapshot_download

    HF_TOKEN = os.getenv("HF_TOKEN", "")
    DATASET_REPO = "juhenes/ngiml"
    DATASET_REVISION = "main"
    if HF_TOKEN:
        login(token=HF_TOKEN)

    snapshot_download(
        repo_id=DATASET_REPO,
        repo_type="dataset",
        local_dir=str(DATA_DIR),
        revision=DATASET_REVISION,
        token=HF_TOKEN or None,
        resume_download=True,
    )
    MANIFEST_PATH = find_or_resolve_manifest(DATA_DIR)

manifest = load_manifest(MANIFEST_PATH)
INFER_NORMALIZATION = str(getattr(manifest, "normalization_mode", "zero_one")).strip().lower()

print("Using manifest:", MANIFEST_PATH)
print("Inference normalization:", INFER_NORMALIZATION)

model, device, ckpt_info = load_model_from_checkpoint(CKPT_PATH)
INFER_THRESHOLD = float(ckpt_info.get("default_threshold", 0.5))
INFER_MAX_SHORT_SIDE = int(ckpt_info.get("max_short_side", 0) or 0)
INFER_THRESHOLD_SOURCE = str(ckpt_info.get("threshold_source", "unknown"))

print("Device:", device)
print("Checkpoint info:", ckpt_info)
print("Inference threshold:", INFER_THRESHOLD)
print("Inference threshold source:", INFER_THRESHOLD_SOURCE)
print("Inference max_short_side:", INFER_MAX_SHORT_SIDE)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from tools.local_infer_helpers import load_image_mask_from_record, predict_probability_map

# Choose which split to inspect for apples-to-apples comparison with training logs.
PREFERRED_SPLIT = "val"  # use "val" to compare against reported validation metrics; set to "test" for held-out testing
FALLBACK_SPLITS = ("val", "test", "train") if PREFERRED_SPLIT == "val" else (PREFERRED_SPLIT, "val", "test", "train")

split_counts = {}
for sample in manifest.samples:
    split_counts[sample.split] = split_counts.get(sample.split, 0) + 1
print("Manifest split counts:", split_counts)

fake_samples = [
    s for s in manifest.samples
    if int(getattr(s, "label", 0)) == 1 or s.mask_path is not None
]
fake_split_counts = {}
for sample in fake_samples:
    fake_split_counts[sample.split] = fake_split_counts.get(sample.split, 0) + 1
print("Fake sample counts by split:", fake_split_counts)

if not fake_samples:
    raise RuntimeError("No fake/manipulated samples found in manifest")

selected_split = None
samples_for_split = []
for split_name in FALLBACK_SPLITS:
    candidates = [s for s in fake_samples if s.split == split_name]
    if candidates:
        selected_split = split_name
        samples_for_split = candidates
        break

if not samples_for_split:
    selected_split = "mixed"
    samples_for_split = fake_samples

MAX_SAMPLES = 10
samples_to_show = samples_for_split[:MAX_SAMPLES]
print(f"Requested split: {PREFERRED_SPLIT}")
print(f"Showing {len(samples_to_show)} fake samples from split: {selected_split}")
print("Using threshold:", INFER_THRESHOLD)
print("Using threshold source:", INFER_THRESHOLD_SOURCE)
print("Using normalization:", INFER_NORMALIZATION)
print("Using max_short_side:", INFER_MAX_SHORT_SIDE)

rows = len(samples_to_show)
fig, axes = plt.subplots(rows, 4, figsize=(16, 3 * rows))
if rows == 1:
    axes = np.expand_dims(axes, axis=0)

col_titles = ["Photo", "Ground Truth", "Prediction", "Overlay (Prediction on Photo)"]
for col, title in enumerate(col_titles):
    axes[0, col].set_title(title)

for i, sample in enumerate(samples_to_show):
    image, gt_mask, high_pass = load_image_mask_from_record(sample, max_short_side=INFER_MAX_SHORT_SIDE)
    pred_prob = predict_probability_map(
        model,
        image,
        device,
        normalization_mode=INFER_NORMALIZATION,
        high_pass=high_pass,
    )

    img_np = image.permute(1, 2, 0).cpu().numpy()
    gt_np = gt_mask[0].cpu().numpy()
    pred_np = pred_prob.numpy()

    pred_bin = (pred_np >= float(INFER_THRESHOLD)).astype(np.float32)
    overlay = img_np.copy()
    overlay[..., 0] = np.clip(overlay[..., 0] + 0.5 * pred_bin, 0, 1)
    overlay[..., 1] = np.clip(overlay[..., 1] * (1.0 - 0.35 * pred_bin), 0, 1)
    overlay[..., 2] = np.clip(overlay[..., 2] * (1.0 - 0.35 * pred_bin), 0, 1)

    axes[i, 0].imshow(img_np)
    axes[i, 1].imshow(gt_np, cmap="gray", vmin=0, vmax=1)
    axes[i, 2].imshow(pred_np, cmap="magma", vmin=0, vmax=1)
    axes[i, 3].imshow(overlay)

    for j in range(4):
        axes[i, j].axis("off")

    axes[i, 0].set_ylabel(f"{sample.dataset}\\n{sample.split}\\n#{i+1}", rotation=0, labelpad=28, va="center")

plt.tight_layout()
plt.show()

In [ ]:
# Full test-split evaluation with the current checkpoint threshold.
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt

TEST_EVAL_SPLIT = "test"
TEST_EVAL_THRESHOLD = float(INFER_THRESHOLD)
PRINT_EVERY = 25

def _metrics_from_counts(tp, tn, fp, fn, eps=1e-6):
    iou = (tp + eps) / (tp + fp + fn + eps)
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)
    f1 = (2.0 * precision * recall) / (precision + recall + eps)
    dice = (2.0 * tp + eps) / (2.0 * tp + fp + fn + eps)
    accuracy = (tp + tn + eps) / (tp + tn + fp + fn + eps)
    return {
        "dice": float(dice),
        "iou": float(iou),
        "f1": float(f1),
        "precision": float(precision),
        "recall": float(recall),
        "accuracy": float(accuracy),
    }

def _confusion_from_totals(tn, fp, fn, tp):
    return np.array([[tn, fp], [fn, tp]], dtype=np.float64)

test_samples = [sample for sample in manifest.samples if sample.split == TEST_EVAL_SPLIT]
if not test_samples:
    raise RuntimeError(f"No samples found for split={TEST_EVAL_SPLIT!r} in manifest {MANIFEST_PATH}")

manifest_mask_paths = sum(1 for sample in test_samples if sample.mask_path is not None)
positive_labeled_samples = sum(1 for sample in test_samples if int(getattr(sample, "label", 0)) == 1)
dataset_counts = {}
for sample in test_samples:
    dataset_counts[sample.dataset] = dataset_counts.get(sample.dataset, 0) + 1

print(f"Evaluating split: {TEST_EVAL_SPLIT}")
print(f"Samples: {len(test_samples)} | manifest mask paths: {manifest_mask_paths} | positive labels: {positive_labeled_samples}")
print("Dataset counts:", dataset_counts)
print(f"Using threshold: {TEST_EVAL_THRESHOLD:.4f} ({INFER_THRESHOLD_SOURCE})")
print("Note: prepared NPZ/tar samples can contain embedded masks even when manifest mask_path is None.")

overall = {"tp": 0.0, "tn": 0.0, "fp": 0.0, "fn": 0.0, "samples": 0}
per_dataset = defaultdict(lambda: {"tp": 0.0, "tn": 0.0, "fp": 0.0, "fn": 0.0, "samples": 0})

model.eval()
for idx, sample in enumerate(test_samples, start=1):
    image, gt_mask, high_pass = load_image_mask_from_record(sample, max_short_side=INFER_MAX_SHORT_SIDE)
    pred_prob = predict_probability_map(
        model,
        image,
        device,
        normalization_mode=INFER_NORMALIZATION,
        high_pass=high_pass,
    ).numpy()

    target = (gt_mask[0].cpu().numpy() >= 0.5).astype(np.float32)
    pred = (pred_prob >= TEST_EVAL_THRESHOLD).astype(np.float32)

    tp = float((pred * target).sum())
    tn = float(((1.0 - pred) * (1.0 - target)).sum())
    fp = float((pred * (1.0 - target)).sum())
    fn = float(((1.0 - pred) * target).sum())

    overall["tp"] += tp
    overall["tn"] += tn
    overall["fp"] += fp
    overall["fn"] += fn
    overall["samples"] += 1

    bucket = per_dataset[str(sample.dataset)]
    bucket["tp"] += tp
    bucket["tn"] += tn
    bucket["fp"] += fp
    bucket["fn"] += fn
    bucket["samples"] += 1

    if idx % PRINT_EVERY == 0 or idx == len(test_samples):
        print(f"Processed {idx}/{len(test_samples)} samples")

overall_metrics = _metrics_from_counts(overall["tp"], overall["tn"], overall["fp"], overall["fn"])
print("Overall test split metrics:")
print({
    "split": TEST_EVAL_SPLIT,
    "samples": int(overall["samples"]),
    "threshold": float(TEST_EVAL_THRESHOLD),
    **overall_metrics,
})

print("Per-dataset test metrics:")
per_dataset_rows = []
for dataset_name in sorted(per_dataset):
    counts = per_dataset[dataset_name]
    metrics = _metrics_from_counts(counts["tp"], counts["tn"], counts["fp"], counts["fn"])
    row = {
        "dataset": dataset_name,
        "samples": int(counts["samples"]),
        "threshold": float(TEST_EVAL_THRESHOLD),
        **metrics,
    }
    per_dataset_rows.append(row)
    print(row)

heatmaps = [(f"Overall {TEST_EVAL_SPLIT}", _confusion_from_totals(overall["tn"], overall["fp"], overall["fn"], overall["tp"]))]
for dataset_name in sorted(per_dataset):
    counts = per_dataset[dataset_name]
    heatmaps.append((
        f"{dataset_name} {TEST_EVAL_SPLIT}",
        _confusion_from_totals(counts["tn"], counts["fp"], counts["fn"], counts["tp"]),
    ))

fig, axes = plt.subplots(1, len(heatmaps), figsize=(5 * len(heatmaps), 4))
if len(heatmaps) == 1:
    axes = [axes]

for ax, (title, confusion_matrix) in zip(axes, heatmaps):
    im = ax.imshow(confusion_matrix, cmap="Blues")
    ax.set_title(f"{title}\nPixel Confusion Matrix")
    ax.set_xticks([0, 1], labels=["Pred Neg", "Pred Pos"])
    ax.set_yticks([0, 1], labels=["True Neg", "True Pos"])
    for row_idx in range(confusion_matrix.shape[0]):
        for col_idx in range(confusion_matrix.shape[1]):
            value = confusion_matrix[row_idx, col_idx]
            ax.text(col_idx, row_idx, f"{value:,.0f}", ha="center", va="center", color="black")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Pixel count")

plt.tight_layout()
plt.show()

print("Overall confusion counts:")
print({
    "tn": float(overall["tn"]),
    "fp": float(overall["fp"]),
    "fn": float(overall["fn"]),
    "tp": float(overall["tp"]),
})

In [ ]:
# Full test-split threshold sweep: overall and per-dataset.
from collections import defaultdict
import numpy as np

if "test_samples" not in globals() or not test_samples:
    raise RuntimeError("Run the full test-split evaluation cell first so test_samples is available.")

def _counts_from_binary(pred_bin, target_bin):
    pred_bin = pred_bin.astype(np.float32)
    target_bin = target_bin.astype(np.float32)
    tp = float((pred_bin * target_bin).sum())
    tn = float(((1.0 - pred_bin) * (1.0 - target_bin)).sum())
    fp = float((pred_bin * (1.0 - target_bin)).sum())
    fn = float(((1.0 - pred_bin) * target_bin).sum())
    return tp, tn, fp, fn

def _metrics_from_counts(tp, tn, fp, fn, eps=1e-6):
    iou = (tp + eps) / (tp + fp + fn + eps)
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)
    f1 = (2.0 * precision * recall) / (precision + recall + eps)
    dice = (2.0 * tp + eps) / (2.0 * tp + fp + fn + eps)
    accuracy = (tp + tn + eps) / (tp + tn + fp + fn + eps)
    return {
        "dice": float(dice),
        "iou": float(iou),
        "f1": float(f1),
        "precision": float(precision),
        "recall": float(recall),
        "accuracy": float(accuracy),
    }

thresholds = np.arange(0.20, 0.82, 0.02, dtype=np.float32)
cached_predictions = []

print("Caching probability maps for full test threshold sweep...")
for idx, sample in enumerate(test_samples, start=1):
    image, gt_mask, high_pass = load_image_mask_from_record(sample, max_short_side=INFER_MAX_SHORT_SIDE)
    pred_prob = predict_probability_map(
        model,
        image,
        device,
        normalization_mode=INFER_NORMALIZATION,
        high_pass=high_pass,
    ).numpy()
    target = (gt_mask[0].cpu().numpy() >= 0.5).astype(np.float32)
    cached_predictions.append({
        "dataset": str(sample.dataset),
        "pred_prob": pred_prob,
        "target": target,
    })
    if idx % PRINT_EVERY == 0 or idx == len(test_samples):
        print(f"Cached {idx}/{len(test_samples)} samples")

overall_rows = []
per_dataset_rows = defaultdict(list)

for threshold in thresholds:
    overall_counts = {"tp": 0.0, "tn": 0.0, "fp": 0.0, "fn": 0.0}
    dataset_counts = defaultdict(lambda: {"tp": 0.0, "tn": 0.0, "fp": 0.0, "fn": 0.0})

    for item in cached_predictions:
        pred_bin = (item["pred_prob"] >= float(threshold)).astype(np.float32)
        tp, tn, fp, fn = _counts_from_binary(pred_bin, item["target"])

        overall_counts["tp"] += tp
        overall_counts["tn"] += tn
        overall_counts["fp"] += fp
        overall_counts["fn"] += fn

        bucket = dataset_counts[item["dataset"]]
        bucket["tp"] += tp
        bucket["tn"] += tn
        bucket["fp"] += fp
        bucket["fn"] += fn

    overall_metrics = _metrics_from_counts(
        overall_counts["tp"], overall_counts["tn"], overall_counts["fp"], overall_counts["fn"]
    )
    overall_rows.append({
        "threshold": float(threshold),
        **overall_metrics,
    })

    for dataset_name, counts in dataset_counts.items():
        metrics = _metrics_from_counts(counts["tp"], counts["tn"], counts["fp"], counts["fn"])
        per_dataset_rows[dataset_name].append({
            "dataset": dataset_name,
            "threshold": float(threshold),
            **metrics,
        })

overall_rows = sorted(overall_rows, key=lambda item: item["iou"], reverse=True)
print("Top overall threshold candidates by full-test IoU:")
for row in overall_rows[:10]:
    print(row)

print("Best threshold per dataset by full-test IoU:")
best_per_dataset = {}
for dataset_name in sorted(per_dataset_rows):
    rows = sorted(per_dataset_rows[dataset_name], key=lambda item: item["iou"], reverse=True)
    best_per_dataset[dataset_name] = rows[0]
    print(rows[0])

best_overall_threshold = overall_rows[0]["threshold"] if overall_rows else INFER_THRESHOLD
print("Current checkpoint threshold:", INFER_THRESHOLD)
print("Best overall threshold on full test:", best_overall_threshold)

In [ ]:
from google.colab import files
from PIL import Image
import io
import math
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from src.data.dataloaders import _compute_high_pass_fallback
from tools.local_infer_helpers import normalize_image_for_inference, resize_for_inference

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No file uploaded.")

filename, file_bytes = next(iter(uploaded.items()))
pil_img = Image.open(io.BytesIO(file_bytes)).convert("RGB")
img_np = np.array(pil_img).astype(np.float32) / 255.0
img = torch.from_numpy(img_np).permute(2, 0, 1)
high_pass = _compute_high_pass_fallback(img)

# Keep spatial size compatible with Swin/decoder and aligned with training short-side cap.
h, w = img.shape[-2], img.shape[-1]
MAX_SIDE_LENGTH = 768
if max(h, w) > MAX_SIDE_LENGTH:
    scale_factor = MAX_SIDE_LENGTH / max(h, w)
    new_h, new_w = int(h * scale_factor), int(w * scale_factor)
else:
    new_h, new_w = h, w

target_h = int(math.ceil(new_h / 32) * 32)
target_w = int(math.ceil(new_w / 32) * 32)
if (target_h, target_w) != (h, w):
    img = F.interpolate(img.unsqueeze(0), size=(target_h, target_w), mode="bilinear", align_corners=False).squeeze(0)
    high_pass = F.interpolate(high_pass.unsqueeze(0), size=(target_h, target_w), mode="bilinear", align_corners=False).squeeze(0)

img, _, high_pass = resize_for_inference(img, mask=None, high_pass=high_pass, max_short_side=INFER_MAX_SHORT_SIDE)
x = normalize_image_for_inference(img, normalization_mode=INFER_NORMALIZATION).unsqueeze(0).to(device)
hp = high_pass.unsqueeze(0).to(device)
with torch.no_grad():
    logits = model(x, target_size=img.shape[-2:], high_pass=hp)[-1]
    pred_prob = torch.sigmoid(logits)[0, 0].detach().cpu().numpy()

img_show = img.permute(1, 2, 0).cpu().numpy()
pred_bin = (pred_prob >= float(INFER_THRESHOLD)).astype(np.float32)

overlay = img_show.copy()
overlay[..., 0] = np.clip(overlay[..., 0] + 0.5 * pred_bin, 0, 1)
overlay[..., 1] = np.clip(overlay[..., 1] * (1.0 - 0.35 * pred_bin), 0, 1)
overlay[..., 2] = np.clip(overlay[..., 2] * (1.0 - 0.35 * pred_bin), 0, 1)

print(f"Uploaded: {filename}")
print("Input tensor shape:", tuple(x.shape))
print("Inference threshold:", INFER_THRESHOLD)
print("Inference normalization:", INFER_NORMALIZATION)
print("Inference max_short_side:", INFER_MAX_SHORT_SIDE)
print("Prediction stats min/mean/max:", float(pred_prob.min()), float(pred_prob.mean()), float(pred_prob.max()))

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.title("Uploaded Photo")
plt.imshow(img_show)
plt.axis("off")

plt.subplot(1, 3, 2)
plt.title("Prediction")
plt.imshow(pred_prob, cmap="magma", vmin=0, vmax=1)
plt.colorbar(fraction=0.046, pad=0.04)
plt.axis("off")

plt.subplot(1, 3, 3)
plt.title("Overlay")
plt.imshow(overlay)
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
%pip install fvcore iopath

In [ ]:
# Model complexity stats (params, FLOPs, MACs)
from tools.local_infer_helpers import get_model_complexity_stats

def _human(n: float | int | None) -> str:
    if n is None:
        return "n/a"
    n = float(n)
    if n >= 1e12:
        return f"{n/1e12:.3f}T"
    if n >= 1e9:
        return f"{n/1e9:.3f}B"
    if n >= 1e6:
        return f"{n/1e6:.3f}M"
    if n >= 1e3:
        return f"{n/1e3:.3f}K"
    return f"{n:.0f}"

# Keep aligned with your training/inference resolution.
PROFILE_INPUT_SIZE = (1, 3, 320, 320)
stats = get_model_complexity_stats(model, input_size=PROFILE_INPUT_SIZE)

print("Model complexity:")
print({
    "input_size": stats["input_size"],
    "total_params": f"{stats['total_params']:,} ({_human(stats['total_params'])})",
    "trainable_params": f"{stats['trainable_params']:,} ({_human(stats['trainable_params'])})",
    "frozen_params": f"{stats['frozen_params']:,} ({_human(stats['frozen_params'])})",
    "flops": f"{_human(stats['flops'])}" if stats.get("flops") is not None else "n/a",
    "macs": f"{_human(stats['macs'])}" if stats.get("macs") is not None else "n/a",
})
if stats.get("flops_error"):
    print("FLOPs note:", stats["flops_error"])